# Task B — retrieval demo
Final seed42 cố định. **Conventional + Retrieval-only Adapter** là mô hình chính; MR là ablation, adapter lấy cảm hứng TME chứ không phải TME nguyên bản. Các nhánh XAA/DVR không có giai đoạn training text encoder bổ sung như nhánh Conventional/Adapter.

GPU → cấu hình → bootstrap/Drive → định nghĩa → setup/index → đối chiếu toàn bộ query → giao diện. Không training. Artifact nguồn chỉ đọc; index có version riêng. Runtime mới cần quyền Drive/Kaggle khi Colab yêu cầu.

Hai chế độ: query official-val (hai caption gốc, target, rank đầy đủ và Hit@10/50) hoặc ảnh + modifier mới (ngoài benchmark, không có ground truth metric). Mặc định Adapter. Top-K5/10, cùng gallery đầy đủ theo category; reference giữ trong gallery. Không dùng target tạo query, không so độ lớn cosine giữa mô hình.


In [ ]:
CFG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103',
 'adapter_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1',
 'image_roots': ['/content/fashioniq_data'],
 'dataset_root': '/content/fashioniq_data',
 'prepare_images_if_missing': True,
 'local_cache': '/content/taskb_retrieval_demo_cache',
 'example_seed': 2026,
 'allow_cpu': False,
 'min_free_disk_gib': 7,
 'smoke_per_category': 1}

CFG.update({
 'mr_root':'/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1',
 'index_root':'/content/drive/MyDrive/ĐATN/data/processed/taskb_demo/faiss_flatip_v1',
 'run_full_validation':True,
 'default_model':'Conventional + Retrieval-only Adapter (TME-inspired)',
})


In [ ]:
# Prepare SDK before dependent imports. Never upgrade torch/CUDA automatically.
import sys,importlib.metadata,subprocess,shutil
PINS={'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.0','faiss-cpu':'1.13.2'}
needed=[]
for package,version in PINS.items():
    module=package.replace('-','_')
    try:installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:installed=None
    loaded=sys.modules.get(module)
    if loaded is not None and (installed!=version or getattr(loaded,'__version__',None)!=version):
        raise RuntimeError('Loaded SDK version differs: '+package+'. Save notebook, restart session once, run from top. No torch/CUDA reinstall.')
    if installed!=version:needed.append(package+'=='+version)
if CFG['prepare_images_if_missing'] and shutil.which('kaggle') is None:needed.append('kaggle')
if needed:subprocess.run([sys.executable,'-m','pip','install','--quiet',*needed],check=True)
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
# Full report retained locally; never print credentials or entire pip logs.
from pathlib import Path
Path('/content/taskb_demo_pip_check.txt').write_text(check.stdout+check.stderr)
from transformers.generation import GenerationMixin
import transformers.models.clip.modeling_clip as clip_module
assert GenerationMixin.__name__ == 'GenerationMixin' and hasattr(clip_module,'CLIPTextTransformer')
from google.colab import drive
drive.mount('/content/drive')
print('SDK imports ready; full pip check report: /content/taskb_demo_pip_check.txt')


In [ ]:
PREP_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95', 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'extraction_contract.json': 'bbd53b94172aa5c07a358a0697163c620a823b297c6fec6209167bdb59d1f0a1', 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21', 'development_gallery.csv': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'development_val.csv': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'development_train.csv': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'feature_views/development_gallery.csv': '283199425c5fc88fb8e063ea28be629e611e44513689e97f22ee3d0ec5a33927', 'feature_views/development_val.csv': 'cbd8d7b72fa7fb4e3c38d64f3bb8ceeeba73be946e9918c5a77c856a42217619', 'feature_views/development_train.csv': '6f7a5e6ab8792dd8fd08d6fea8c472a9d9b56b59af38ba146467d797f8a98ff0', 'official_train_queries.csv': '539c53d61e9b3d01bf010d089ffa16cef2cd437229feee4a475884586188fd11', 'official_val_queries.csv': '79ea1887f7d5cfd85174754833e4fe2bb6bb74c61c5dd49563cb03b58d9b327c', 'official_val_gallery.csv': '58e274df22b52ed5eb3dbc3428ad2966f56b2fb6202cabc0c194f53e2bb8a3d9'}
EXTRACTION_CONTRACT = {'implementation': 'fashioniq-preparation-v1.0.0', 'encoder_id': 'patrickjohncyh/fashion-clip', 'revision': '7e3ba62ce16b379a1ab479346b66f192e76f51b7', 'encoder_weight_sha256': '7b2e2766a4372167f9db95876b897dcecfe1ca5963f2bf620b3a8393f3ad6366', 'processor': {'_processor_class': None, 'do_resize': True, 'size': {'shortest_edge': 224}, 'resample': 3, 'do_center_crop': True, 'crop_size': {'height': 224, 'width': 224}, 'do_rescale': True, 'rescale_factor': 0.00392156862745098, 'do_normalize': True, 'image_mean': [0.48145466, 0.4578275, 0.40821073], 'image_std': [0.26862954, 0.26130258, 0.27577711], 'do_convert_rgb': True, 'image_processor_type': 'CLIPImageProcessor'}, 'tokenizer': {'class': 'CLIPTokenizer', 'vocab_sha256': 'f9190aead40a965a09db1ba2e556dccac0eda8423cb4926bcd994722899ec68e', 'special_tokens': {'bos_token': '<|startoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, 'max_length': 77}, 'caption_policy': 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1', 'truncation_policy': 'error_if_over_77_preserve_all_source_rows', 'dtype': 'float32', 'attention_backend': 'eager', 'autocast': False, 'tf32': False, 'libraries': {'torch': '2.11.0+cu130', 'transformers': '4.57.1', 'tokenizers': '0.22.1', 'huggingface-hub': '0.36.0', 'numpy': '2.4.6', 'pandas': '2.2.3', 'Pillow': '11.3.0'}, 'device': 'cuda', 'device_name': 'Tesla T4', 'cuda': '13.0', 'batch_size': 32, 'feature_shard_size': 256, 'reference': 'vision_model.last_hidden_state INCLUDING CLS; no projection or L2', 'text': 'text_model.last_hidden_state, [77,512], no projection/normalization + binary attention_mask', 'target': 'visual_projection(vision_model.pooler_output) then L2, [512]', 'repeat_acceptance': {'atol': 6.103515625e-05, 'rtol': 6.103515625e-05, 'predeclared': True, 'note': 'Repeat same input and execution contract; not a historical feature acceptance threshold.'}}
PAIR = {'adapter': {'encoder_checkpoint_sha256': 'b63517597e4adfd0e863c75795fb675dcedd80bd64f59131a367d0d704c0e780', 'epochs': 9, 'format': 'native', 'kind': 'adapter', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_42/adapter/final.pt', 'phase': 'final', 'seed': 42, 'sha256': '03dec75cee1045f0c65fafeb880b148f85ee0769728704136142c8001a308221', 'signature': 'f290d9066e6fcde04fd6a0d076a3d76fb78229a4a06b5e4315a8c73c8f3abfd6'}, 'encoder': {'encoder_checkpoint_sha256': None, 'epochs': 3, 'format': 'native', 'kind': 'encoder', 'path': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_full_experiment/FashionIQ/conventional_retrieval_adapter_full_v1/final/seed_42/encoder/final.pt', 'phase': 'final', 'seed': 42, 'sha256': 'b63517597e4adfd0e863c75795fb675dcedd80bd64f59131a367d0d704c0e780', 'signature': '2e4215e5c984b0ea50330759e1da760adf50cace67e86de46715a73a983010fe'}, 'final_lock_sha256': '44489b7638dbb8c05c78e49840c4153115b8753305dde4be32a641f26ae00fc4', 'official_val_evaluated': False, 'seed': 42, 'text_audit_sha': '00a5272e27aed5c346417af386189c796d72a3007087195abdd748dcd2d740e3', 'image_audit_sha': 'f397f4f38d12051ef7bfa0468085eea3810ec2cdc91237beaa53d0e56d1eafb9', 'conventional_pred_sha': '199ef5ab957dff8a077d79a74fc4b37af3a3ff912c2602d74e6a1553c29ab1e4', 'adapter_pred_sha': '3ca6ba53520ce039c6f0aeb88b7c2d2cba80f3bead6a864a3430fc811b0162ff', 'text_index_sha': '47fb662c206efda9059d8592f2f417259262f133191c913cdc054924bfbda62f', 'pool_index_sha': '27067a74f850b703b7eb655069621cbeb03fa9a621dd848767c5650590962a0c'}
HEADS = {'XAA baseline': {'model_id': 'xaa_noft', 'folder': '/content/drive/MyDrive/ĐATN/data/processed/taskb_training/FashionIQ/heads_v1_20261103_ioopt_v1/final/xaa_noft/seed_42', 'config_sha': 'dad6ad2b336ab6fdda5fc654ac521f945efc4278f475e269fcf2f707ee2afca0', 'checkpoint_sha': 'a60a47c8d7fbc14d0ba785c45458dfa01d7d0c9c9095238e2e7898c126e762aa', 'pred_sha': '8b8126e0e519514216b930d60f9245984ce31ac410b45793c246860d1236546c'}, 'XAA + DVR-MR': {'model_id': 'xaa_mr', 'folder': '/content/drive/MyDrive/ĐATN/data/processed/taskb_training/FashionIQ/heads_v1_20261103_ioopt_v1/final/xaa_mr/seed_42', 'config_sha': '9d551961c231461b5a705d9eb1e13d91c8fbbf314a56a7d2d3675dac58990c67', 'checkpoint_sha': '61ebaf74c46273cd0c2b9cce1caad9f98a5a7fb918933e306befca32fe1f93d4', 'pred_sha': '7b58a1b541e08bbd30d05956fcb1350c7e35676e5cea51b495766fdc2553baeb'}}
CAPTION_POLICY = 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1'
CATEGORIES = ('dress', 'shirt', 'toptee')
ADAPTER_LABEL = 'Conventional + Retrieval-only Adapter (TME-inspired)'
DVR_HIDDEN = 256
DVR_ALPHA = 1.0
MR_PINS = {'report_sha256': '60bc8d19747b5b598d0ff6474353a8d364d950c9d7b4470653cbf2dd08f01928', 'summary_sha256': 'a6056e184568ac3f481beb2223781b5fb1c248ad529bdcbf47061c17dfb3cdd4', 'report_receipts': {'/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/final/seed_42/completion.json': 'dbd9f548a3784074214337f2e6ed464a98cf5fbc9c3e63513478e02792a43c2d', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/final/seed_43/completion.json': '4c47845df0a3ee92e08c946c8c5dd30a4025a264988fc076d48273f92a4ce505', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/final/seed_44/completion.json': '8bf2051051e19ea8e030862fb51991c75960af31be752bcd4e7605135c075a6a', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/final_mr_lock.json': '6234cd246eae2ff481101e1dc072faba4f8f423361005f375a9fd69a69255ece', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/official_val_summary.json': 'a6056e184568ac3f481beb2223781b5fb1c248ad529bdcbf47061c17dfb3cdd4', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/official_val_v2/seed_42/adapter/result.json': '53b1baee0b7abfa242823e1e60169de22313958dfb60cdf37def6400332657e8', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/official_val_v2/seed_42/adapter_mr/result.json': '9a17e2818ca965cbd876b8c4958ca5281544a2102dc4e85d36d61f8a66555f84', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/official_val_v2/seed_43/adapter/result.json': '4d077dac7783947e69a80f390c0b19809e71fdd141ab6b848cc03bbb9f3465f1', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/official_val_v2/seed_43/adapter_mr/result.json': 'ff1489c6053cae14607907f1673dfbc1b49f3cb673c5dfcf91751f9d8b6511f3', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/official_val_v2/seed_44/adapter/result.json': 'ca8cc6b29d8edd4a9e887a7a858a0fb9f4016dac1037bf02d07841eec026b533', '/content/drive/MyDrive/ĐATN/data/processed/adapter_mr_final/FashionIQ/adapter_mr_final_v1/official_val_v2/seed_44/adapter_mr/result.json': 'faa72c8ee2e80deb153a59b7b79f4aba1f9dc0cfc4ca72181a113b64357979a7'}}
MR_LABEL = "Conventional + Adapter + MR (TME-inspired)"


In [ ]:
import os,json,hashlib,math,copy,shutil,html
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from IPython.display import display,HTML,clear_output

def require(ok,message,*unused):
    if not ok:raise RuntimeError(message)
def canonical(x):return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode()
def fingerprint(x):return hashlib.sha256(canonical(x)).hexdigest()
def file_hash(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
def read_json(p):return json.loads(Path(p).read_text())
def verified_json(p,sha):require(file_hash(p)==sha,'JSON checksum changed: '+str(p));return read_json(p)
def safe_child(root,rel):
    root=Path(root).resolve();p=(root/rel).resolve();require(root in p.parents,'Path escapes root');return p

def read_checkpoint(p,sha):
    require(file_hash(p)==sha,'Checkpoint SHA mismatch: '+str(p))
    # Only the owner-verified, pinned binary is deserialized. No downloaded third-party checkpoint.
    return torch.load(p,map_location='cpu',weights_only=False)


def tensor_hash(module):
    h = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        h.update(name.encode()); h.update(str(value.dtype).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def finite(x, name):
    require(bool(torch.isfinite(x).all()), 'NaN/Inf: '+name); return x

def unit(x):
    finite(x, 'unnormalized vector')
    require(bool((x.float().norm(dim=-1)>1e-8).all()), 'Zero retrieval vector')
    return F.normalize(x.float(), dim=-1)

def eos_positions(ids, mask, eos_token_id):
    require(ids.shape==mask.shape and bool(((mask==0)|(mask==1)).all()), 'Token/mask shape or values invalid')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()), 'Empty text token sequence')
    expected=torch.arange(ids.shape[1],device=ids.device)[None,:]<lengths[:,None]
    require(torch.equal(expected,mask.bool()), 'Tokenizer must right-pad contiguous sequences')
    # pad_token == EOS for CLIP; intersect with attention_mask and select first real EOS.
    is_eos=(ids==eos_token_id)&mask.bool()
    require(bool(is_eos.any(-1).all()), 'EOS absent from unpadded tokens')
    pos=is_eos.long().argmax(-1)
    require(bool((pos==lengths-1).all()), 'EOS must close each untruncated modifier')
    return pos

def text_masks(mask, dtype):
    b,n=mask.shape; neg=torch.finfo(dtype).min
    padding=torch.zeros((b,1,n,n),device=mask.device,dtype=dtype).masked_fill(~mask[:,None,None,:].bool(),neg)
    causal=torch.full((n,n),neg,dtype=dtype,device=mask.device).triu(1)
    return padding,causal[None,None].expand(b,1,n,n)

class ResidualAdapter(nn.Module):
    def __init__(self,projection_weight,text_dim=512,vision_dim=768,width=128,heads=4):
        super().__init__();require(width%heads==0,'Adapter width/head mismatch')
        self.register_buffer('projection_weight',projection_weight.detach().clone())
        self.text_norm=nn.LayerNorm(text_dim);self.text_down=nn.Linear(text_dim,width)
        self.reference_norm=nn.LayerNorm(vision_dim);self.reference_down=nn.Linear(vision_dim,width)
        self.cross_attention=nn.MultiheadAttention(width,heads,dropout=0.,batch_first=True)
        self.ri_mlp=nn.Sequential(nn.Linear(width,width),nn.GELU(),nn.Linear(width,text_dim))
        self.ta_mlp=nn.Sequential(nn.Linear(text_dim,width),nn.GELU(),nn.Linear(width,text_dim))
        self.gate_ri=nn.Parameter(torch.zeros(()));self.gate_ta=nn.Parameter(torch.zeros(()))
    def forward(self,reference_features,text_features,mask,eos,return_aux=False,gates_off=False):
        require(text_features.ndim==3 and mask.shape==text_features.shape[:2] and eos.shape==(len(text_features),),'Adapter input shape')
        require(bool(((mask==0)|(mask==1)).all()) and bool((eos==mask.sum(-1)-1).all()),'Mask/EOS invalid')
        require(bool((eos>=1).all()) and bool((mask.sum(-1)<=mask.shape[1]).all()),'Invalid sequence length')
        require(torch.equal(mask.bool(),torch.arange(mask.shape[1],device=mask.device)[None,:]<mask.sum(-1)[:,None]),'Mask must be contiguous right padding')
        valid=mask.unsqueeze(-1).to(text_features.dtype)
        x=self.text_norm(text_features);q=self.text_down(x);kv=self.reference_down(self.reference_norm(reference_features['tokens']))
        ri=self.cross_attention(q,kv,kv,need_weights=False)[0];ri=self.ri_mlp(ri)*valid
        ta=self.ta_mlp(x)*valid
        gri=self.gate_ri.new_zeros(()) if gates_off else torch.tanh(self.gate_ri)
        gta=self.gate_ta.new_zeros(()) if gates_off else torch.tanh(self.gate_ta)
        enhanced=(text_features+gri*ri+gta*ta)*valid # no LN/MLP on the base stream
        index=torch.arange(len(eos),device=eos.device)
        text=unit(F.linear(enhanced[index,eos],self.projection_weight))
        query=unit(text+unit(reference_features['pooled'])) # reference composed exactly once
        if return_aux:
            student=F.linear((text_features+ta)[index,eos],self.projection_weight)
            return query,student,enhanced
        return query
    def encode_query(self,reference_features,text_features,mask,eos):
        return self(reference_features,text_features,mask,eos)
    def enhanced_text_sequence(self,reference_features,text_features,mask,eos):
        _,_,h=self(reference_features,text_features,mask,eos,return_aux=True)
        return h,mask,eos

class ConventionalOnly(nn.Module):
    """Exactly the source conventional path; no RI/TA/full-depth modules are created."""
    def __init__(self,text,projection,eos_id):
        super().__init__();self.text=copy.deepcopy(text);self.projection=copy.deepcopy(projection);self.eos_id=int(eos_id)
        self.requires_grad_(True);self.text.embeddings.requires_grad_(False)
    def encode_text(self,modifier):
        ids,mask=modifier['input_ids'],modifier['attention_mask'];eos=eos_positions(ids,mask,self.eos_id)
        h=self.text.embeddings(input_ids=ids);padding,causal=text_masks(mask,h.dtype);valid=mask.unsqueeze(-1).to(h.dtype)
        for block in self.text.encoder.layers:h=block(h,attention_mask=padding,causal_attention_mask=causal)[0]*valid
        return self.text.final_layer_norm(h)*valid,mask,eos
    def encode_query(self,reference,modifier):
        h,mask,eos=self.encode_text(modifier);text=self.projection(h[torch.arange(len(h),device=h.device),eos])
        return unit(unit(text)+unit(reference['pooled']))
    def freeze(self):self.requires_grad_(False);self.eval();return self

class CachedConventional(nn.Module):
    def __init__(self,projection):super().__init__();self.register_buffer('projection_weight',projection.detach().clone())
    def forward(self,ref,text,mask,eos):
        return unit(unit(F.linear(text[torch.arange(len(eos),device=eos.device),eos],self.projection_weight))+unit(ref['pooled']))
    def encode_query(self,*args):return self(*args)

class XAAFusionModule(nn.Module):
    def __init__(self, text_dim=512, img_dim=768, embed_dim=512, num_heads=8, dropout=0.1):
        super().__init__()
        self.ref_proj = nn.Linear(img_dim, embed_dim)
        self.ref_ln   = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                                          batch_first=True, dropout=dropout)
        self.out_ln = nn.LayerNorm(embed_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def forward(self, text_hidden, text_mask, reference_hidden):
        ref = self.ref_ln(self.ref_proj(reference_hidden))              # [B,50,512]
        att, _ = self.attn(query=text_hidden, key=ref, value=ref)       # [B,77,512]
        x = self.out_ln(text_hidden + att)                              # residual + LN
        m = text_mask.unsqueeze(-1)                                     # [B,77,1]
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1.0)               # masked mean
        return F.normalize(pooled, dim=-1)

class ModifierGuidedRefinement(nn.Module):
    def __init__(self, dim=512, hidden=DVR_HIDDEN, alpha=DVR_ALPHA):
        super().__init__()
        self.alpha = float(alpha)
        self.v_proj = nn.Linear(dim, hidden)       # W_v
        self.t_proj = nn.Linear(dim, hidden)       # W_t
        self.act = nn.GELU()
        self.out = nn.Linear(hidden, 1)            # gate head
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)   # identity tại khởi tạo

    def forward(self, v, text_hidden, text_mask):
        # v: [B,N,D] | text_hidden: [B,L,D] | text_mask: [B,L] (1 = token thật)
        m = text_mask.unsqueeze(-1).to(text_hidden.dtype)
        t = (text_hidden * m).sum(1) / m.sum(1).clamp(min=1.0)                  # masked mean, bỏ padding   [B,D]
        h = self.act(self.v_proj(v) + self.t_proj(t).unsqueeze(1))              # [B,N,hidden]
        gate = 1.0 + self.alpha * torch.tanh(self.out(h).squeeze(-1))          # [B,N]  ∈ (1-α, 1+α)
        return v * gate.unsqueeze(-1), gate

class SelfGuidedRefinement(nn.Module):
    def __init__(self, dim=512, hidden=DVR_HIDDEN, alpha=DVR_ALPHA):
        super().__init__()
        self.alpha = float(alpha)
        self.v_proj = nn.Linear(dim, hidden)       # W_v
        self.c_proj = nn.Linear(dim, hidden)       # W_c (ngữ cảnh thị giác toàn ảnh)
        self.act = nn.GELU()
        self.out = nn.Linear(hidden, 1)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, v):                           # CHỈ nhận image tokens — không có đường vào cho text
        ctx = v.mean(dim=1)                                                     # [B,D]
        h = self.act(self.v_proj(v) + self.c_proj(ctx).unsqueeze(1))            # [B,N,hidden]
        gate = 1.0 + self.alpha * torch.tanh(self.out(h).squeeze(-1))          # [B,N]
        return v * gate.unsqueeze(-1), gate

class XAADVR(nn.Module):
    def __init__(self, use_mr=False, use_sr=False, text_dim=512, img_dim=768, embed_dim=512, num_heads=8,
                 dropout=0.1, dvr_hidden=DVR_HIDDEN, dvr_alpha=DVR_ALPHA):
        super().__init__()
        self.use_mr, self.use_sr = bool(use_mr), bool(use_sr)
        # --- XAA cơ bản: cùng thứ tự khởi tạo với XAAFusionModule ---
        self.ref_proj = nn.Linear(img_dim, embed_dim)
        self.ref_ln   = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, batch_first=True, dropout=dropout)
        self.out_ln = nn.LayerNorm(embed_dim)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
        # --- DVR: tạo sau, rồi trả RNG về trạng thái ngay sau khi tạo XAA ---
        _rng = torch.get_rng_state()
        self.mr = ModifierGuidedRefinement(embed_dim, dvr_hidden, dvr_alpha) if self.use_mr else None
        self.sr = SelfGuidedRefinement(embed_dim, dvr_hidden, dvr_alpha) if self.use_sr else None
        torch.set_rng_state(_rng)

    def forward(self, text_hidden, text_mask, reference_hidden, return_gates=False):
        ref = self.ref_ln(self.ref_proj(reference_hidden))              # Image Projection  [B,50,512]
        gates = {}
        if self.mr is not None:
            ref, gates["mr"] = self.mr(ref, text_hidden, text_mask)    # MR
        if self.sr is not None:
            ref, gates["sr"] = self.sr(ref)                            # SR (sau MR)
        att, _ = self.attn(query=text_hidden, key=ref, value=ref)       # XAA không đổi
        x = self.out_ln(text_hidden + att)
        m = text_mask.unsqueeze(-1)
        pooled = (x * m).sum(1) / m.sum(1).clamp(min=1.0)
        out = F.normalize(pooled, dim=-1)
        return (out, gates) if return_gates else out
class ReadonlyCache:
    def __init__(self,root,role,ids,local_root,kind=False):
        self.root=Path(root);self.role=role;self.index=read_json(self.root/'index.json');self.key=self.index['fingerprint'];self.kind=kind
        if kind:
            require(self.index['kind']==role,'Feature kind mismatch');self.ids=self.index['ids'];self.tail=self.index['shape_tail'];self.dtype=self.index['dtype'];self.step=self.index['shard_size']
            require(read_json(self.root.parent/(role+'_id2idx.json'))=={v:i for i,v in enumerate(self.ids)},'Feature row mapping mismatch')
        else:
            require(fingerprint(self.index['contract'])==self.key and self.root.name==self.key,'Cache contract fingerprint mismatch')
            spec=self.index['specs'][role];self.ids=spec['ids'];self.tail=spec['shape'];self.dtype=spec['dtype'];self.step=self.index['shard_size']
        self.map={v:i for i,v in enumerate(self.ids)};require(len(self.map)==len(self.ids) and set(ids)<=set(self.map),'Missing/duplicate feature IDs')
        self.paths={};self.loaded=OrderedDict();self.local=Path(local_root)/self.key/role;self.local.mkdir(parents=True,exist_ok=True)
        needed=sorted({self.map[v]//self.step*self.step for v in ids})
        for start in needed:
            end=min(start+self.step,len(self.ids));name=f'{start:07d}_{end:07d}';folder=self.root if kind else self.root/role;p=folder/(name+'.npy');marker=read_json(p.with_suffix('.json'))
            expected={'fingerprint':self.key,'kind' if kind else 'role':role,'ids_sha256':fingerprint(self.ids[start:end]),'shape':[end-start,*self.tail],'dtype':self.dtype}
            require(all(marker.get(k)==v for k,v in expected.items()),'Committed shard identity mismatch: '+str(p))
            require(file_hash(p)==marker['sha256'],'Source shard corrupt: '+str(p));dest=self.local/p.name
            if not dest.exists() or file_hash(dest)!=marker['sha256']:
                tmp=dest.with_suffix('.partial')
                with p.open('rb') as src,tmp.open('wb') as dst:shutil.copyfileobj(src,dst,8*1024*1024)
                require(file_hash(tmp)==marker['sha256'],'Local copy corrupt');os.replace(tmp,dest)
            a=np.load(dest,mmap_mode='r',allow_pickle=False);require(list(a.shape)==expected['shape'] and str(a.dtype)==self.dtype and np.isfinite(a).all(),'Invalid shard values/header')
            if role in ('target','reference_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Non-unit image embedding')
            if role=='mask':require(np.isin(a,[0,1]).all(),'Invalid mask')
            self.paths[start]=dest
    def take(self,ids):
        values=[]
        for v in ids:
            row=self.map[v];start=row//self.step*self.step;require(start in self.paths,'ID not preflighted')
            if start not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[start]=np.load(self.paths[start],mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(start);values.append(np.array(self.loaded[start][row-start],copy=True))
        return np.stack(values)

def rank_scores(scores,target_pos):
    require(np.isfinite(scores).all(),'Nonfinite ranking scores')
    order=np.argsort(-scores,kind='stable');rank=1+int((scores>scores[target_pos]).sum())+int(((scores==scores[target_pos])&(np.arange(len(scores))<target_pos)).sum())
    return rank,order

def prediction_table(path,sha):
    require(file_hash(path)==sha,'Benchmark CSV changed: '+str(path));p=pd.read_csv(path,dtype={'query_id':str,'reference_image_id':str,'target_image_id':str})
    keys=['query_id','category','reference_image_id','target_image_id'];require(len(p)==len(Q) and p.query_id.is_unique,'Benchmark query coverage mismatch')
    require(len(p.merge(Q[keys],on=keys,validate='one_to_one'))==len(Q),'Benchmark query/source mapping mismatch')
    require(np.isfinite(p.target_rank).all() and (p.target_rank==p.target_rank.astype(int)).all(),'Invalid saved ranks')
    for c in CATEGORIES:require(p.loc[p.category==c,'target_rank'].between(1,int((G.category==c).sum())).all(),'Saved ranks outside category gallery')
    return p.set_index('query_id').target_rank.astype(int)

def setup_demo():
    global Q,G,MODELS,STATUS,PRED,STORES,GALLERIES,RESULT_CACHE,DEVICE,TRAINED_ENCODER
    require(torch.cuda.is_available() or CFG['allow_cpu'],'Select a GPU runtime, or explicitly allow CPU inference')
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.set_default_dtype(torch.float32);torch.backends.cudnn.benchmark=False
    require(not torch.is_autocast_enabled(),'Disable inherited autocast: benchmark uses FP32')
    local=Path(CFG['local_cache']);local.mkdir(parents=True,exist_ok=True);require(shutil.disk_usage(local).free>CFG['min_free_disk_gib']*1024**3,'Insufficient disk for validated local cache')
    prep=Path(CFG['preparation_root']);meta=verified_json(prep/'feature_metadata.json',PREP_ANCHORS['feature_metadata.json']);protocol=verified_json(prep/'protocol_config.json',PREP_ANCHORS['protocol_config.json'])
    require(read_json(prep/'extraction_contract.json')==EXTRACTION_CONTRACT,'Extraction contract mismatch');require(protocol['caption_policy']==CAPTION_POLICY and protocol['reference_policy']=='retain in gallery (FashionIQ)','Protocol mismatch')
    for name in ('official_val_queries.csv','official_val_gallery.csv'):require(file_hash(prep/name)==PREP_ANCHORS[name],'Manifest checksum mismatch: '+name)
    Q=pd.read_csv(prep/'official_val_queries.csv',dtype=str,keep_default_na=False);G=pd.read_csv(prep/'official_val_gallery.csv',dtype=str,keep_default_na=False);G.gallery_index=G.gallery_index.astype(int)
    require(len(Q)==6016 and Q.query_id.is_unique,'Require all6016 official-val queries')
    require(Q.groupby('category').size().to_dict()=={'dress':2017,'shirt':2038,'toptee':1961},'Query category counts differ')
    require(G.groupby('category').size().to_dict()=={'dress':3817,'shirt':6346,'toptee':5373},'Full gallery counts differ')
    for r in Q.itertuples():require(r.modification_text==r.caption_1.strip('.?, ').capitalize()+' and '+r.caption_2.strip('.?, '),'Caption policy changed')
    for c in CATEGORIES:
        g=G[G.category==c];require(g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))),'Gallery order/duplicates')
        require(set(Q[Q.category==c].target_image_id)<=set(g.image_id),'Target coverage missing');require(fingerprint(g.image_id.tolist())==protocol['id_order_sha256']['official_val_gallery'][c],'Gallery ID order differs')
    feature_root=prep/'features'/meta['fingerprint'];STORES={};MODELS={};STATUS={};PRED={};RESULT_CACHE={}
    # Image reference/target inputs are shared by every comparison, including full candidate galleries.
    for role,ids,shape in [('reference',Q.reference_image_id,[50,768]),('target',G.image_id,[512])]:
        require(file_hash(feature_root/role/'index.json')==meta['store_index_sha256'][role],'Image index changed')
        reader=ReadonlyCache(feature_root/role,role,ids,local,True);require(reader.tail==shape and reader.key==meta['fingerprint'] and reader.dtype=='float32','Image feature contract');STORES[role]=reader
    GALLERIES={c:(G[G.category==c].sort_values('gallery_index',kind='stable').image_id.tolist(),None) for c in CATEGORIES}
    for c,(ids,_) in list(GALLERIES.items()):
        a=STORES['target'].take(ids);GALLERIES[c]=(ids,a/np.linalg.norm(a,axis=-1,keepdims=True))
    for label,pin in HEADS.items():
        try:
            folder=Path(pin['folder']);cfg=verified_json(folder/'training_config.json',pin['config_sha']);require(cfg['phase']=='final' and cfg['seed']==42 and cfg['epoch_budget']==8 and cfg['model']==pin['model_id'],'Wrong head checkpoint config/phase/seed')
            state=read_checkpoint(folder/'final_checkpoint.pt',pin['checkpoint_sha']);require(state['phase']=='final' and state['finished'] and state['epoch']==8 and state['config']==cfg and state['signature']==fingerprint(cfg),'Head payload provenance differs')
            a=cfg['architecture'];args=dict(text_dim=a['text_dim'],img_dim=a['reference_dim'],embed_dim=a['embed_dim'],num_heads=a['num_heads'],dropout=a['attention_dropout'])
            model=XAAFusionModule(**args) if pin['model_id']=='xaa_noft' else XAADVR(**args,use_mr=True,use_sr=False,dvr_hidden=a['dvr_hidden'],dvr_alpha=a['dvr_alpha'])
            model.load_state_dict(state['model_state'],strict=True);del state;MODELS[label]=model.eval().requires_grad_(False).to(DEVICE);PRED[label]=prediction_table(folder/'per_query_results.csv',pin['pred_sha']);STATUS[label]='ready; strict final seed42 loading passed'
        except FileNotFoundError as e:MODELS.pop(label,None);STATUS[label]='missing_assets: '+str(e)
        except Exception as e:MODELS.pop(label,None);STATUS[label]='invalid/incompatible: '+str(e)
    if any(x in MODELS for x in HEADS):
        for role,shape in [('text',[77,512]),('mask',[77])]:
            require(file_hash(feature_root/role/'index.json')==meta['store_index_sha256'][role],'Text index mismatch')
            STORES['pre_'+role]=ReadonlyCache(feature_root/role,role,Q.query_id,local,True);require(STORES['pre_'+role].tail==shape and STORES['pre_'+role].key==meta['fingerprint'] and STORES['pre_'+role].dtype==('int64' if role=='mask' else 'float32'),'Pretrained text shape/fingerprint mismatch')
    try:
        root=Path(CFG['adapter_root']);asset=PAIR['encoder'];state=read_checkpoint(asset['path'],asset['sha256']);require(state['phase']=='final' and state['seed']==42 and state['kind']=='encoder' and state['finished'] and state['epoch']==3,'Wrong final encoder')
        from transformers import CLIPConfig,CLIPTokenizer
        from transformers.models.clip.modeling_clip import CLIPTextTransformer
        config=CLIPConfig.from_pretrained(EXTRACTION_CONTRACT['encoder_id'],revision=EXTRACTION_CONTRACT['revision']);config.text_config._attn_implementation='eager'
        tokenizer=CLIPTokenizer.from_pretrained(EXTRACTION_CONTRACT['encoder_id'],revision=EXTRACTION_CONTRACT['revision'])
        # Source Adapter Full builds ConventionalOnly with tokenizer.eos_token_id (49407), not legacy config EOS2.
        encoder=ConventionalOnly(CLIPTextTransformer(config.text_config),nn.Linear(512,512,bias=False),tokenizer.eos_token_id)
        encoder.load_state_dict(state['model'],strict=True);encoder.freeze();del state
        audit=verified_json(root/'benchmark/seed_42/text_cache_audit.json',PAIR['text_audit_sha']);require(audit['passed'] and audit['queries']==6016 and audit['phase']=='benchmark' and audit['encoder_checkpoint']==asset,'Wrong benchmark text cache')
        textroot=root/'text_cache'/audit['fingerprint'];index=verified_json(textroot/'index.json',PAIR['text_index_sha']);ct=index['contract'];require(ct['encoder_checkpoint_sha256']==asset['sha256'] and ct['encoder_weights_sha256']==tensor_hash(encoder) and ct['phase']=='benchmark' and ct['caption_policy']==CAPTION_POLICY and ct['validation_sha256']==PREP_ANCHORS['official_val_queries.csv'],'Text cache encoder/manifest mismatch')
        for role in ('text','mask','eos','input_ids'):
            STORES['trained_'+role]=ReadonlyCache(textroot,role,Q.query_id,local)
            require(STORES['trained_'+role].ids==sorted(Q.query_id.tolist()),'Trained text ID order differs')
            require(STORES['trained_'+role].tail=={'text':[77,512],'mask':[77],'eos':[],'input_ids':[77]}[role],'Trained cache shape differs')
            require(STORES['trained_'+role].dtype==('float32' if role=='text' else 'int64'),'Trained cache dtype differs')
        tokenizer=CLIPTokenizer.from_pretrained(EXTRACTION_CONTRACT['encoder_id'],revision=EXTRACTION_CONTRACT['revision'])
        for start in range(0,len(Q),128):
            rows=Q.iloc[start:start+128];ids=rows.query_id.tolist();tok=tokenizer(rows.modification_text.tolist(),padding='max_length',max_length=77,truncation=False,return_tensors='np')
            a=STORES['trained_input_ids'].take(ids);m=STORES['trained_mask'].take(ids);e=STORES['trained_eos'].take(ids);h=STORES['trained_text'].take(ids)
            require(a.shape[1]==77 and np.array_equal(a,tok['input_ids']) and np.array_equal(m,tok['attention_mask']),'Token/caption/row alignment mismatch')
            require(np.array_equal(eos_positions(torch.from_numpy(a),torch.from_numpy(m),tokenizer.eos_token_id).numpy(),e),'EOS mismatch');require(np.count_nonzero(h[m==0])==0 and (np.linalg.norm(h[np.arange(len(ids)),e],axis=-1)>1e-8).all(),'Invalid padding/EOS feature')
        ia=verified_json(root/'benchmark/seed_42/image_feature_audit.json',PAIR['image_audit_sha']);require(ia['passed'] and ia['preparation_feature_fingerprint']==meta['fingerprint'],'Image source mismatch')
        verified_json(root/'shared_reference_pool'/ia['reference_pool']/'index.json',PAIR['pool_index_sha'])
        STORES['pool']=ReadonlyCache(root/'shared_reference_pool'/ia['reference_pool'],'reference_pooled',Q.reference_image_id,local)
        require(STORES['pool'].index['contract']['image_contract']==EXTRACTION_CONTRACT and STORES['pool'].ids==sorted(set(Q.reference_image_id)),'Reference pool contract/ID coverage mismatch')
        MODELS['Conventional-only']=CachedConventional(encoder.projection.weight).eval().to(DEVICE)
        STATUS['Conventional-only']='ready; same final encoder seed42';PRED['Conventional-only']=prediction_table(root/'benchmark/seed_42/conventional/per_query_results.csv',PAIR['conventional_pred_sha'])
        try:
            aa=PAIR['adapter'];s=read_checkpoint(aa['path'],aa['sha256']);require(s['phase']=='final' and s['seed']==42 and s['kind']=='adapter' and s['epoch']==9 and s['finished'] and s['binding']['encoder_asset']['sha256']==asset['sha256'],'Wrong adapter checkpoint/encoder')
            model=ResidualAdapter(encoder.projection.weight);model.load_state_dict(s['model'],strict=True);require(torch.equal(model.projection_weight,encoder.projection.weight),'Adapter projection mismatch');MODELS[ADAPTER_LABEL]=model.eval().requires_grad_(False).to(DEVICE);del s
            PRED[ADAPTER_LABEL]=prediction_table(root/'benchmark/seed_42/adapter/per_query_results.csv',PAIR['adapter_pred_sha']);STATUS[ADAPTER_LABEL]='ready; final9epoch seed42, TME-inspired'
        except Exception as e:MODELS.pop(ADAPTER_LABEL,None);STATUS[ADAPTER_LABEL]='unavailable: '+str(e)
        TRAINED_ENCODER=encoder.to(DEVICE)
    except Exception as e:
        for label in ('Conventional-only',ADAPTER_LABEL):MODELS.pop(label,None);STATUS[label]='missing/invalid pair source: '+str(e)
    display(pd.DataFrame([{'model':k,'status':v} for k,v in STATUS.items()]))
    require(MODELS,'No model has complete validated assets; see paths/status')
    print('Ready:',len(MODELS),'models | 6016 queries | category galleries:',{k:len(v[0]) for k,v in GALLERIES.items()})


def eligible(category,mode='Tất cả'):
    ids=Q.loc[Q.category==category,'query_id'].tolist()
    if mode=='Tất cả':return ids
    require('Conventional-only' in MODELS and ADAPTER_LABEL in MODELS,'Bộ lọc cần đủ hai mô hình cùng encoder và per-query results')
    a=PRED[ADAPTER_LABEL].loc[ids];b=PRED['Conventional-only'].loc[ids]
    mask={'Adapter tốt hơn':a<b,'Adapter kém hơn':a>b,'Cả hai Top10':(a<=10)&(b<=10),'Cả hai ngoài Top50':(a>50)&(b>50)}[mode]
    return [v for v in ids if bool(mask.loc[v])]

def sample_examples(mode='Tất cả',per_category=1):
    rng=np.random.default_rng(CFG['example_seed']);result=[]
    for c in CATEGORIES:
        pool=sorted(eligible(c,mode));result.extend(rng.choice(pool,min(per_category,len(pool)),replace=False).tolist())
    return result

def image_panel(ax,image_id,title):
    ax.axis('off');paths=IMAGE_PATHS.get(image_id,set())
    try:
        require(len(paths)==1,'Thiếu ảnh' if not paths else 'ID có nhiều đường dẫn ảnh, cần kiểm tra')
        with Image.open(next(iter(paths))) as im:im.load();ax.imshow(im.convert('RGB'))
    except Exception as e:ax.text(.5,.5,'Không hiển thị được\n'+str(e)[:100],ha='center',va='center',wrap=True)
    ax.set_title(title+'\n'+image_id,fontsize=9)

import tempfile,zipfile,contextlib,stat,subprocess
from pathlib import PurePosixPath
KAGGLE_DATASET = 'binhhuunguyen/fashioniq'
KAGGLE_CREDENTIAL_DIR = Path('/root/.kaggle')
_KAGGLE_USE_UPLOADED_JSON = False
class EnvironmentPreflightBlocked(RuntimeError):
    pass

class DatasetPreparationError(RuntimeError):
    pass

def _dataset_inside(path, root):
    path, root = Path(path).resolve(), Path(root).resolve()
    return path == root or root in path.parents

def ensure_kaggle_cli():
    executable=shutil.which('kaggle')
    if executable is None:
        raise EnvironmentPreflightBlocked('Kaggle CLI chưa có trong PATH. Chạy Bootstrap trước; không cài package sau khi đã import benchmark.')
    return executable

def _kaggle_command(arguments, timeout=120):
    executable = ensure_kaggle_cli()
    try:
        with contextlib.ExitStack() as stack:
            env = None  # Thử nguyên trạng credentials/env hiện có trước.
            if _KAGGLE_USE_UPLOADED_JSON:
                # Tránh token/config cũ lấn JSON mới upload; chỉ dùng một bản sao runtime tạm.
                isolated = stack.enter_context(tempfile.TemporaryDirectory(prefix='cli-', dir=KAGGLE_CREDENTIAL_DIR))
                isolated_json = Path(isolated)/'kaggle.json'
                shutil.copyfile(KAGGLE_CREDENTIAL_DIR/'kaggle.json', isolated_json)
                isolated_json.chmod(0o600)
                env = os.environ.copy()
                for key in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
                    env.pop(key, None)
                env['KAGGLE_CONFIG_DIR'] = isolated
            completed = subprocess.run([executable, *arguments], capture_output=True, text=True,
                                       env=env, timeout=timeout, stdin=subprocess.DEVNULL)
    except subprocess.TimeoutExpired:
        raise DatasetPreparationError('Lệnh Kaggle hết thời gian; dừng trước benchmark, kiểm tra mạng rồi chạy lại.') from None
    except OSError:
        raise DatasetPreparationError('Không chạy được Kaggle CLI trong runtime hoặc chưa có JSON xác thực đã upload.') from None
    # Không in stdout/stderr của CLI: có thể chứa thông tin xác thực hoặc URL ký sẵn.
    if completed.returncode == 0:
        return 'ok'
    message = (completed.stdout or '') + '\n' + (completed.stderr or '')
    lower = message.lower()
    if any(marker in lower for marker in ('401', '403', 'unauthorized', 'forbidden', 'credential',
                                         'authenticate', 'authentication', 'kaggle.json', 'kaggle_api_token')):
        return 'auth_required'
    raise DatasetPreparationError(
        f'Kaggle CLI thất bại (exit {completed.returncode}); kiểm tra mạng/quyền truy cập dataset. '
        'Không xuất log thô để tránh lộ thông tin xác thực.') from None

def upload_kaggle_json_if_needed():
    # Credentials chỉ nằm trong filesystem tạm của runtime, không đưa vào CFG/Drive/notebook output.
    from google.colab import files
    credential_dir = KAGGLE_CREDENTIAL_DIR
    credential_dir.mkdir(parents=True, exist_ok=True, mode=0o700)
    destination = credential_dir/'kaggle.json'
    previous_cwd = Path.cwd()
    uploaded = None
    try:
        with tempfile.TemporaryDirectory(prefix='upload-', dir=credential_dir) as private_upload:
            os.chdir(private_upload)  # files.upload() không ghi bản gốc vào Drive/current working directory.
            print('Chọn một JSON Legacy API Credentials có username và key (tên file có thể bất kỳ).')
            uploaded = files.upload()
            if not isinstance(uploaded, dict) or len(uploaded) != 1:
                raise DatasetPreparationError('Cần upload đúng một file JSON xác thực Kaggle.')
            actual_name = next(iter(uploaded))  # Dùng đúng tên files.upload() trả về, không giả định kaggle.json.
            try:
                payload = json.loads(uploaded[actual_name].decode('utf-8'))
            except (ValueError, UnicodeError, AttributeError):
                raise DatasetPreparationError('File upload không phải JSON UTF-8 hợp lệ.') from None
            if not isinstance(payload, dict) or not all(
                    isinstance(payload.get(key), str) and payload[key].strip() for key in ('username', 'key')):
                raise DatasetPreparationError('JSON cần có hai trường chuỗi không rỗng: username và key.')
            # Chỉ ghi các trường cần thiết; không lưu tên file upload hoặc credential vào artifact benchmark.
            secure_json = {'username': payload['username'].strip(), 'key': payload['key'].strip()}
            fd, temp_name = tempfile.mkstemp(prefix='.credentials-', dir=credential_dir)
            try:
                os.fchmod(fd, 0o600)
                with os.fdopen(fd, 'w', encoding='utf-8') as stream:
                    json.dump(secure_json, stream)
                os.replace(temp_name, destination)
                destination.chmod(0o600)
            finally:
                if Path(temp_name).exists():
                    Path(temp_name).unlink()
    finally:
        os.chdir(previous_cwd)
        if isinstance(uploaded, dict):
            uploaded.clear()
    print('Đã lưu JSON xác thực trong runtime với quyền 600; không in nội dung.')

def _extract_dataset_archive(archive, data_root):
    """Giải nén dưới DATA_ROOT; tiếp tục bản dở bằng cách bỏ qua file có bytes giống hệt."""
    root = Path(data_root).resolve()
    try:
        with zipfile.ZipFile(archive) as bundle:
            for member in bundle.infolist():
                relative = PurePosixPath(member.filename)
                if (relative.is_absolute() or '..' in relative.parts or '\\' in member.filename
                        or stat.S_ISLNK(member.external_attr >> 16)):
                    raise DatasetPreparationError('Archive có đường dẫn hoặc symlink không an toàn; dừng giải nén.')
                if any(part.startswith('._') or part == '__MACOSX' for part in relative.parts):
                    continue
                target = root.joinpath(*relative.parts)
                if not _dataset_inside(target, root):
                    raise DatasetPreparationError('Archive trỏ ra ngoài DATA_ROOT.')
                if member.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                    continue
                target.parent.mkdir(parents=True, exist_ok=True)
                if target.exists():
                    if not target.is_file() or target.is_symlink():
                        raise DatasetPreparationError(f'Xung đột dữ liệu có sẵn: {target}')
                    with bundle.open(member) as source:
                        expected_hash = hashlib.sha256()
                        for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
                            expected_hash.update(block)
                    if file_hash(target) != expected_hash.hexdigest():
                        raise DatasetPreparationError(f'File có sẵn khác archive; không ghi đè: {target}')
                    continue
                # File .partial không được nhận là JPG/JSON đầy đủ sau khi runtime bị ngắt.
                temp = target.with_name(target.name + '.extracting')
                with bundle.open(member) as source, temp.open('wb') as output:
                    shutil.copyfileobj(source, output)
                os.replace(temp, target)
    except DatasetPreparationError:
        raise
    except Exception:
        raise DatasetPreparationError('Giải nén FashionIQ thất bại hoặc archive hỏng; không tiếp tục benchmark.') from None

def download_and_extract_dataset(data_root):
    global _KAGGLE_USE_UPLOADED_JSON
    root = Path(data_root)
    root.mkdir(parents=True, exist_ok=True)
    downloads = root/'.kaggle_downloads'
    downloads.mkdir(exist_ok=True)
    # Thư mục mới cho mỗi lần tải; không dùng ZIP dở từ lần thất bại trước.
    with tempfile.TemporaryDirectory(prefix='fashioniq-', dir=downloads) as work:
        print('Tải binhhuunguyen/fashioniq; thời gian phụ thuộc mạng và dung lượng dataset...')
        outcome = _kaggle_command(['datasets', 'download', '-d', KAGGLE_DATASET, '-p', work, '--quiet'], timeout=3600)
        if outcome == 'auth_required' and not _KAGGLE_USE_UPLOADED_JSON:
            print('Kaggle yêu cầu xác thực khi tải (listing có thể công khai); cần JSON xác thực.')
            upload_kaggle_json_if_needed()
            _KAGGLE_USE_UPLOADED_JSON = True
            outcome = _kaggle_command(['datasets', 'download', '-d', KAGGLE_DATASET, '-p', work, '--quiet'], timeout=3600)
        if outcome != 'ok':
            raise DatasetPreparationError('Kaggle từ chối tải sau xác thực; kiểm tra quyền truy cập/điều khoản dataset. Không tiếp tục benchmark.')
        archives = sorted(Path(work).glob('*.zip'))
        if len(archives) != 1 or not zipfile.is_zipfile(archives[0]):
            raise DatasetPreparationError('Tải chưa tạo đúng một ZIP hợp lệ; không giải nén/không chạy benchmark.')
        _extract_dataset_archive(archives[0], root)
def index_demo_images():
    result={}
    for folder in CFG['image_roots']:
        p=Path(folder)
        if p.is_dir():
            for parent,_,names in os.walk(p):
                for name in names:
                    if not name.startswith('._') and Path(name).suffix.lower() in ('.jpg','.jpeg'):
                        result.setdefault(Path(name).stem,set()).add(str((Path(parent)/name).resolve()))
    return result

def prepare_demo_images():
    global IMAGE_PATHS
    IMAGE_PATHS=index_demo_images();required=set(G.image_id)|set(Q.reference_image_id)
    missing=required-set(IMAGE_PATHS)
    if missing and CFG['prepare_images_if_missing']:
        root=Path(CFG['dataset_root']);root.mkdir(parents=True,exist_ok=True)
        require(shutil.disk_usage(root).free>7*1024**3,'Need at least 7 GiB free for dataset download/extraction; preserve existing features')
        print('Chưa có đủ ảnh official-val:',len(missing),'ID. Chuẩn bị dataset Kaggle trong /content; không ghi Drive.')
        download_and_extract_dataset(root)
        IMAGE_PATHS=index_demo_images();missing=required-set(IMAGE_PATHS)
        require(not missing,'Dataset đã giải nén vẫn thiếu ảnh official-val; không tự loại ID: '+', '.join(sorted(missing)[:8]))
    print('Ảnh indexed:',len(IMAGE_PATHS),'| official-val ID thiếu:',len(missing),'| ID có nhiều đường dẫn:',sum(len(v)>1 for k,v in IMAGE_PATHS.items() if k in required))
    if missing:print('Thiếu ảnh chỉ ảnh hưởng hiển thị. Bật CFG[prepare_images_if_missing] hoặc thêm đúng image_roots; gallery/rank không thay đổi.')


## Setup và xác minh
Checksum/strict loading, full mapping/shard coverage được kiểm trước inference. IndexFlatIP lưu mapping, vector SHA, normalization, protocol và checkpoint provenance; chỉ chia sẻ khi cùng contract và bytes. Nguồn đổi → version mới, không ghi đè nguồn.

FAISS so với exact cosine NumPy trên6016 query/model: điểm, Top10, rank đầy đủ và từng Hit@1/5/10/50; sai khác số học/tie được giữ trong JSON. Không sửa rank để khớp. Historical seed42 có một rank95→94 ngoàiTop50 đã ghi nhận; cần đọc lại chẩn đoán của lượt hiện tại. Validation không đồng nghĩa đã native re-encode toàn dataset.

Nguồn: [FAISS cosine](https://github.com/facebookresearch/faiss/wiki/MetricType-and-distances). Chi tiết kiểm thử và manifest được lưu riêng; không khẳng định FAISS nhanh hơn khi chưa đo.


In [ ]:
def validate_frozen_text(text,mask,eos):
    require(text.ndim==3 and text.shape[-1]==512 and text.shape[1]==77,'Text must be [B,77,512]')
    require(text.dtype==torch.float32 and mask.dtype==eos.dtype==torch.int64,'FP32 text / int64 mask and EOS required')
    require(mask.shape==text.shape[:2] and eos.shape==(len(text),),'Mask/EOS shape mismatch')
    require(bool(((mask==0)|(mask==1)).all()),'Mask must be binary')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()) and torch.equal(eos,lengths-1),'EOS must end untruncated text')
    require(torch.equal(mask.bool(),torch.arange(mask.shape[1],device=mask.device)[None,:]<lengths[:,None]),'Mask must right-pad contiguously')
    finite(text,'frozen text')
    require(torch.count_nonzero(text[mask==0]).item()==0,'Frozen text padding must be zero')
    return text,mask,eos

@torch.no_grad()
def reduce_frozen_text(text,mask,eos,projection_weight):
    """Reduced immutable input, not a new encoder. No target parameter is accepted."""
    validate_frozen_text(text,mask,eos)
    require(projection_weight.shape==(512,512) and projection_weight.dtype==torch.float32,'Frozen text projection schema')
    t=unit(F.linear(text[torch.arange(len(text),device=text.device),eos],projection_weight))
    m=mask.unsqueeze(-1).to(text.dtype);guidance=(text*m).sum(1)/m.sum(1).clamp(min=1.0)
    return {'text_embedding':t.detach(),'guidance':finite(guidance,'MR text guidance').detach()}

@torch.no_grad()
def project_reference_tokens(tokens,vision_post_layernorm,visual_projection):
    """Use frozen Fashion-CLIP modules from the verified preparation revision.
    Source pooled reference is unit(project(postLN(CLS))). Per-token vectors stay
    unnormalized: MR receives the frozen projected tokens; normalization occurs
    only after combining the reference residual with the existing CLS vector.
    """
    require(tokens.ndim==3 and tokens.shape[1:]==(50,768) and tokens.dtype==torch.float32,'Reference tokens must be FP32 [B,50,768]')
    require(not vision_post_layernorm.training and not visual_projection.training,'Frozen vision modules must be eval')
    require(all(not p.requires_grad for m in (vision_post_layernorm,visual_projection) for p in m.parameters()),'Frozen vision modules unexpectedly trainable')
    output=visual_projection(vision_post_layernorm(finite(tokens,'raw reference tokens')))
    require(output.shape==(len(tokens),50,512) and output.dtype==torch.float32,'Projected MR reference schema')
    return finite(output,'projected reference tokens').detach()

class ResidualMRCore(nn.Module):
    """Only MR is trainable. Inputs are checkpoint-bound immutable cache rows."""
    def __init__(self,hidden=256,alpha=1.0):
        super().__init__();self.mr=ModifierGuidedRefinement(512,hidden,alpha)
    def forward(self,projected_tokens,pooled,text_embedding,guidance,mr_off=False):
        b=len(projected_tokens)
        require(projected_tokens.shape==(b,50,512) and pooled.shape==text_embedding.shape==guidance.shape==(b,512),'Cached MR input shape mismatch')
        require(all(x.dtype==torch.float32 for x in (projected_tokens,pooled,text_embedding,guidance)),'Cached MR inputs must be FP32')
        require(len({x.device for x in (projected_tokens,pooled,text_embedding,guidance)})==1,'Cached MR input devices differ')
        for name,x in [('projected_tokens',projected_tokens),('pooled',pooled),('text_embedding',text_embedding),('guidance',guidance)]:finite(x,name)
        require(bool(torch.allclose(text_embedding.norm(dim=-1),torch.ones(b,device=text_embedding.device),rtol=1e-5,atol=1e-5)),'Cached text embedding not unit')
        v=projected_tokens.detach();base=pooled.detach();text=text_embedding.detach();guide=guidance.detach()
        if mr_off:delta=torch.zeros_like(base)
        else:
            gated,_=self.mr(v,guide[:,None,:],torch.ones((b,1),dtype=torch.int64,device=guide.device))
            delta=(gated-v).mean(1)
        return unit(text+unit(base+delta))
    def encode_query(self,reference_features,text_embedding,guidance,mr_off=False):
        return self(reference_features['mr_tokens'],reference_features['pooled'],text_embedding,guidance,mr_off)

class ResidualMRQuery(nn.Module):
    """Native full-sequence wrapper for extraction/preflight parity with cached core.
    C passes trained conventional hidden states. D passes enhanced hidden states
    from the already-trained frozen Adapter. No Adapter/encoder weights train.
    """
    def __init__(self,projection_weight,hidden=256,alpha=1.0):
        super().__init__();self.register_buffer('projection_weight',projection_weight.detach().clone())
        self.core=ResidualMRCore(hidden,alpha)
    def forward(self,reference_features,text,mask,eos,mr_off=False):
        reduced=reduce_frozen_text(text,mask,eos,self.projection_weight)
        return self.core(reference_features['mr_tokens'],reference_features['pooled'],**reduced,mr_off=mr_off)
    def encode_query(self,*args,**kwargs):return self(*args,**kwargs)

@torch.no_grad()
def frozen_adapter_text(adapter,reference_features,text,mask,eos):
    require(not adapter.training and all(not p.requires_grad for p in adapter.parameters()),'Adapter must be frozen/eval')
    validate_frozen_text(text,mask,eos)
    enhanced,returned_mask,returned_eos=adapter.enhanced_text_sequence(reference_features,text,mask,eos)
    require(torch.equal(returned_mask,mask) and torch.equal(returned_eos,eos),'Adapter changed mask/EOS')
    validate_frozen_text(enhanced,mask,eos)
    return enhanced.detach()

import time, uuid, io, platform
import faiss

def array_sha(a):
    a=np.ascontiguousarray(a)
    return hashlib.sha256(str(a.dtype).encode()+str(a.shape).encode()+a.tobytes()).hexdigest()

def immutable_json(path,value):
    path=Path(path);body=canonical(value)
    if path.exists():require(path.read_bytes()==body,'Immutable output differs: '+str(path));return
    path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.'+uuid.uuid4().hex+'.partial');tmp.write_bytes(body);os.replace(tmp,path)

def normalize_np(a):
    a=np.ascontiguousarray(a,dtype=np.float32);norm=np.linalg.norm(a,axis=-1,keepdims=True)
    require(a.ndim==2 and a.shape[1]==512 and np.isfinite(a).all() and (norm>1e-8).all(),'Invalid retrieval vectors')
    return np.ascontiguousarray(a/norm)

class ExactIPIndex:
    """All candidates searched. Only exactly equal FAISS scores use gallery-index tie-breaking."""
    def __init__(self,category,ids,vectors,contract,root):
        self.ids=list(ids);self.vectors=np.ascontiguousarray(vectors,dtype=np.float32)
        require(len(ids)==len(set(ids))==len(vectors) and np.isfinite(vectors).all(),'Index ID/vector coverage invalid')
        require(np.allclose(np.linalg.norm(vectors,axis=1),1,rtol=1e-5,atol=1e-5),'Index vectors must be L2 normalized')
        self.contract={**contract,'schema':'taskb-faiss-flatip-v1','category':category,'gallery_id_sha256':fingerprint(ids),'vectors_sha256':array_sha(self.vectors),'dimension':512,'dtype':'float32','count':len(ids),'metric':'L2-normalized inner product','faiss_version':faiss.__version__,'faiss_distribution_version':__import__('importlib.metadata',fromlist=['version']).version('faiss-cpu'),'architecture':platform.machine(),'faiss_compile_options':faiss.get_compile_options(),'tie_policy':'descending native FAISS score, ascending locked gallery index for exact ties'}
        self.key=fingerprint(self.contract);self.folder=Path(root)/self.key;meta=self.folder/'metadata.json'
        if self.folder.exists():
            require(meta.is_file(),'Incomplete index version; preserve it and inspect: '+str(self.folder))
            old=read_json(meta);require(old['contract']==self.contract and old['fingerprint']==self.key,'Index contract differs')
            for name,sha in old['receipts'].items():require(file_hash(self.folder/name)==sha,'Index file corrupt: '+name)
            require(read_json(self.folder/'mapping.json')==self.ids,'Index row mapping differs')
            self.index=faiss.read_index(str(self.folder/'index.faiss'))
        else:
            # Atomic directory publication; no previous version is overwritten.
            parent=self.folder.parent;parent.mkdir(parents=True,exist_ok=True);stage=parent/('.building-'+uuid.uuid4().hex);stage.mkdir()
            self.index=faiss.IndexFlatIP(512);self.index.add(self.vectors)
            faiss.write_index(self.index,str(stage/'index.faiss'));(stage/'mapping.json').write_bytes(canonical(ids))
            receipts={name:file_hash(stage/name) for name in ('index.faiss','mapping.json')}
            (stage/'metadata.json').write_bytes(canonical({'contract':self.contract,'fingerprint':self.key,'receipts':receipts}))
            os.rename(stage,self.folder)
        require(type(self.index).__name__=='IndexFlatIP' and self.index.metric_type==faiss.METRIC_INNER_PRODUCT and self.index.d==512 and self.index.ntotal==len(ids),'Loaded FAISS schema differs')
        require(np.array_equal(self.index.reconstruct_n(0,len(ids)),self.vectors),'FAISS stored vectors differ from locked source')
    def search_all(self,queries):
        queries=np.ascontiguousarray(queries,dtype=np.float32)
        require(queries.ndim==2 and queries.shape[1]==512 and np.isfinite(queries).all(),'Invalid search queries')
        require(np.allclose(np.linalg.norm(queries,axis=1),1,rtol=1e-5,atol=1e-5),'Queries must be L2 normalized')
        started=time.perf_counter();scores,indices=self.index.search(queries,self.index.ntotal)
        # Preserve FAISS scores; explicit stable order for truly equal scores.
        for i in range(len(queries)):
            require(len(np.unique(indices[i]))==len(self.ids) and set(indices[i])==set(range(len(self.ids))),'FAISS returned wrong row mapping')
            order=np.lexsort((indices[i],-scores[i]));indices[i]=indices[i,order];scores[i]=scores[i,order]
        return scores,indices,time.perf_counter()-started

def verify_final_result_sources():
    root=Path(CFG['mr_root']);report=verified_json(root/'test_report.json',MR_PINS['report_sha256']);summary=verified_json(root/'official_val_summary.json',MR_PINS['summary_sha256'])
    require(report['passed'] and summary['passed'] and summary['complete_three_seeds'] and summary['mr_epochs']==1,'Final MR report incomplete')
    # Re-read every model receipt, source checkpoint and metric; do not use reference figures as results.
    full_lock=read_json(Path(CFG['adapter_root'])/'final_training_lock.json')
    require(fingerprint(full_lock)==PAIR['final_lock_sha256'] and full_lock['encoder_epochs']==3 and full_lock['adapter_epochs']==9 and not full_lock['official_val_metric_used'],'Source encoder/Adapter lock differs')
    receipts=report['receipt_map'];verified=[]
    for seed in (42,43,44):
        path=root/'final'/f'seed_{seed}'/'completion.json';done=verified_json(path,receipts[str(path)]);binding=done['binding']
        require(done['completed'] and done['passed'] and done['seed']==seed and binding['phase']=='final' and binding['kind']=='mr_residual_final' and binding['variant']=='D' and done['train_rows']==18000,'Final provenance differs')
        require(binding['mr_epochs']==1 and binding['data']['train_sha256']==PREP_ANCHORS['official_train_queries.csv'] and binding['source_full_final_lock_sha256']==PAIR['final_lock_sha256'],'Final budget/data/source lock differs')
        for role in ('encoder','adapter'):
            asset=binding['source_'+role];require(asset['seed']==seed and asset['phase']=='final' and file_hash(asset['path'])==asset['sha256'],'Source checkpoint missing/changed')
        require(file_hash(root/'final'/f'seed_{seed}'/'final.pt')==done['file_receipts'][str(root/'final'/f'seed_{seed}'/'final.pt')],'Final MR binary changed')
        records={}
        for label in ('adapter','adapter_mr'):
            p=root/'official_val_v2'/f'seed_{seed}'/label/'result.json';r=verified_json(p,receipts[str(p)]);require(r['passed'] and r['seed']==seed and r['phase']=='benchmark' and r['metrics']['n_query']==6016,'Benchmark receipt invalid')
            for path,sha in r['file_receipts'].items():require(file_hash(path)==sha,'Benchmark result checksum changed')
            pred_path=p.parent/'per_query_results.csv';pred=prediction_table(pred_path,r['file_receipts'][str(pred_path)])
            means={c:{f'R@{k}':float((pred.loc[Q[Q.category==c].query_id]<=k).mean()*100) for k in (1,5,10,50)} for c in CATEGORIES}
            for category,metrics in means.items():
                for metric,value in metrics.items():require(abs(value-r['metrics']['per_category'][category][metric])<1e-10,'Per-query metric readback differs')
            mean=float(np.mean([(metrics['R@10']+metrics['R@50'])/2 for metrics in means.values()]))
            require(abs(mean-r['metrics']['Mean'])<1e-10,'Mean readback differs')
            saved=next(x for x in summary['per_seed'] if x['seed']==seed and x['model']==label);require(abs(saved['Mean']-mean)<1e-10,'Summary/model result mismatch')
            records[label]=r
        verified.append({'seed':seed,'encoder':binding['source_encoder'],'adapter':binding['source_adapter'],'MR':{'path':str(root/'final'/f'seed_{seed}'/'final.pt'),'sha256':done['file_receipts'][str(root/'final'/f'seed_{seed}'/'final.pt')]},'binding':binding,'benchmark':{label:{'path':str(root/'official_val_v2'/f'seed_{seed}'/label/'result.json'),'metrics':r['metrics'],'cache':r['cache']} for label,r in records.items()}})
    lock=verified_json(root/'final_mr_lock.json',receipts[str(root/'final_mr_lock.json')]);require(fingerprint(lock)==summary['lock_sha256'] and lock['mr_epochs']==1,'MR lock fingerprint differs')
    values={label:np.array([r['Mean'] for r in summary['per_seed'] if r['model']==label]) for label in ('adapter','adapter_mr')}
    require(all(len(v)==3 for v in values.values()),'Three-seed coverage missing')
    exact=[]
    for label,v in [*values.items(),('paired_delta',values['adapter_mr']-values['adapter'])]:
        rec=next(r for r in summary['mean_sample_std'] if r['model']==label and r['metric']=='Mean');require(abs(float(v.mean())-rec['mean'])<1e-12 and abs(float(v.std(ddof=1))-rec['sample_std'])<1e-12,'Summary mean/std differs');exact.append(rec)
    return root,report,verified,lock,exact

@torch.inference_mode()
def setup_mr_and_indexes():
    global MR_RESULT,INDEXES,QUERY_VECTORS,DEMO_REPORT,MR_REDUCED,MR_VISION,HANDOVER
    root=Path(CFG['mr_root'])
    try:root,report,verified,lock,exact=verify_final_result_sources()
    except FileNotFoundError as error:
        report=None;verified=[];lock=None;exact=[]
        STATUS[MR_LABEL]='missing_assets: '+str(error)
    receipts=report['receipt_map'] if report is not None else {}
    # MR optional; never relabel another model when assets are unavailable.
    try:
        if report is None:raise FileNotFoundError('MR result source incomplete: '+str(root))
        require(ADAPTER_LABEL in MODELS,'Adapter unavailable, so MR branch cannot load')
        done=read_json(root/'final/seed_42/completion.json');MR_RESULT=verified_json(root/'official_val_v2/seed_42/adapter_mr/result.json',receipts[str(root/'official_val_v2/seed_42/adapter_mr/result.json')])
        ck=read_checkpoint(root/'final/seed_42/final.pt',done['file_receipts'][str(root/'final/seed_42/final.pt')]);require(ck['binding']==done['binding'] and ck['signature']==fingerprint(done['binding']) and ck['finished'] and ck['epoch']==1 and ck['step']==563,'Final MR checkpoint state differs')
        require(done['binding']['source_encoder']==PAIR['encoder'] and done['binding']['source_adapter']==PAIR['adapter'],'Demo MR wrong seed/final sources')
        model=ResidualMRCore(hidden=lock['recipe']['hidden'],alpha=lock['recipe']['alpha']);model.load_state_dict(ck['model'],strict=True);MODELS[MR_LABEL]=model.eval().requires_grad_(False).to(DEVICE);STATUS[MR_LABEL]='ready: final seed42, shared MR lock1 epoch, ablation only'
        cache=MR_RESULT['cache'];vr=root/'cache/vision'/cache['vision'];rr=root/'cache/benchmark/seed_42'/cache['reduced'];refs=Q.reference_image_id.tolist()
        MR_VISION={role:ReadonlyCache(vr,role,refs,Path(CFG['local_cache'])/'mr') for role in ('mr_tokens','reference_pooled')}
        MR_REDUCED={role:ReadonlyCache(rr,role,Q.query_id,Path(CFG['local_cache'])/'mr') for role in ('D_text_embedding','D_guidance')}
        contract=read_json(rr/'index.json')['contract'];require(contract['phase']=='benchmark' and contract['seed']==42 and contract['encoder_sha256']==PAIR['encoder']['sha256'] and contract['adapter_sha256']==PAIR['adapter']['sha256'] and contract['text_index_sha256']==PAIR['text_index_sha'] and contract['vision_cache']==cache['vision'] and not contract['target_used_in_reduction'],'Reduced cache checkpoint/text contract differs')
        require(np.array_equal(MR_VISION['reference_pooled'].take(refs),STORES['pool'].take(refs)),'MR/source pooled reference mismatch')
        path=root/'official_val_v2/seed_42/adapter_mr/per_query_results.csv';PRED[MR_LABEL]=prediction_table(path,MR_RESULT['file_receipts'][str(path)])
    except FileNotFoundError as e:STATUS[MR_LABEL]='missing_assets: '+str(e);MODELS.pop(MR_LABEL,None)
    INDEXES={};QUERY_VECTORS={};DEMO_REPORT={'scope':'actual runtime inference; no training','runtime':{'python':platform.python_version(),'torch':torch.__version__,'numpy':np.__version__,'faiss':faiss.__version__,'device':str(DEVICE)},'models':STATUS,'full_faiss_validation':{},'sources_read_directly':verified,'locked_statistics':exact}
    meta=read_json(Path(CFG['preparation_root'])/'feature_metadata.json');protocol=read_json(Path(CFG['preparation_root'])/'protocol_config.json')
    shared_indexes={}
    for label in MODELS:
        INDEXES[label]={}
        for c,(ids,gallery) in GALLERIES.items():
            # Same per-model normalization count as its locked evaluator.
            vectors=normalize_np(gallery) if label in HEADS else gallery
            contract={'producer':'frozen pretrained Fashion-CLIP visual projection','extraction_contract':EXTRACTION_CONTRACT,'target_index_sha256':meta['store_index_sha256']['target'],'feature_fingerprint':meta['fingerprint'],'manifest_sha256':PREP_ANCHORS['official_val_gallery.csv'],'protocol_sha256':PREP_ANCHORS['protocol_config.json'],'reference_policy':protocol['reference_policy'],'normalization_count_after_preparation':2 if label in HEADS else 1}
            shared_key=fingerprint({'contract':contract,'category':c,'ids':ids,'vectors':array_sha(vectors)})
            if shared_key not in shared_indexes:shared_indexes[shared_key]=ExactIPIndex(c,ids,vectors,contract,CFG['index_root'])
            INDEXES[label][c]=shared_indexes[shared_key]
    HANDOVER={'schema':'taskb-handover-v1','main_model':'Conventional + Retrieval-only Adapter (TME-inspired)','MR_role':'ablation with marginal paired development-locked official-val change; not original TME or a SOTA claim','seeds':[42,43,44],'demo_seed':42,'protocol':protocol,'manifest_sha256':{k:v for k,v in PREP_ANCHORS.items() if k.startswith('official_')},'sources':verified,'exact_statistics':exact,'head_demo_sources':HEADS,'feature_contract':EXTRACTION_CONTRACT,'indexes':{label:{c:{'folder':str(x.folder),'fingerprint':x.key,'contract':x.contract} for c,x in index.items()} for label,index in INDEXES.items()},'historical_rank_caveat':'shirt:val:000693 historical95/current94; unchanged reported hit cutoffs. New FAISS differences are reported separately.','verification':'checkpoint byte hashes and strict state loading seed42; metadata/binary hashes and benchmark receipts other seeds; no training in this demo task','private_artifacts':'Drive owner permission required; no redistribution or sharing changes'}
    register_query_index_bindings()
    print('Validated models:',list(MODELS),'| Index contracts:',len({x.key for d in INDEXES.values() for x in d.values()}))

@torch.inference_mode()
def dataset_query_vectors(label,category):
    key=(label,category)
    if key in QUERY_VECTORS:return QUERY_VECTORS[key]
    frame=Q[Q.category==category];parts=[];started=time.perf_counter();batch=256 if label in HEADS else 32
    for start in range(0,len(frame),batch):
        rows=frame.iloc[start:start+batch];qids=rows.query_id.tolist();refs=rows.reference_image_id.tolist();t=lambda reader,ids:torch.from_numpy(reader.take(ids)).to(DEVICE)
        if label==MR_LABEL:
            q=MODELS[label](t(MR_VISION['mr_tokens'],refs),t(MR_VISION['reference_pooled'],refs),t(MR_REDUCED['D_text_embedding'],qids),t(MR_REDUCED['D_guidance'],qids))
        elif label in HEADS:q=MODELS[label](t(STORES['pre_text'],qids),t(STORES['pre_mask'],qids),t(STORES['reference'],refs))
        else:q=MODELS[label].encode_query({'tokens':t(STORES['reference'],refs),'pooled':t(STORES['pool'],refs)},t(STORES['trained_text'],qids),t(STORES['trained_mask'],qids),t(STORES['trained_eos'],qids))
        parts.append(q.cpu().numpy())
    q=normalize_np(np.concatenate(parts));QUERY_VECTORS[key]=(q,frame.query_id.tolist(),time.perf_counter()-started);return QUERY_VECTORS[key]

def float32_dot_bound(q,g):
    # IEEE-754 dot-product accumulation bound, derived from dimension, not tuned to observed ranks.
    u=np.finfo(np.float32).eps/2;gamma=(q.shape[1]*u)/(1-q.shape[1]*u)
    return 2*gamma*(np.abs(q).astype('float64')@np.abs(g).astype('float64').T)

def validate_faiss_all():
    order=[label for label in (ADAPTER_LABEL,MR_LABEL) if label in MODELS]+[label for label in MODELS if label not in (ADAPTER_LABEL,MR_LABEL)]
    for label in order:
        report={'queries':0,'score_max_abs':0.,'rank_differences':[],'top10_differences':[],'historical_rank_differences':[],'metric_hit_differences':{str(k):0 for k in (1,5,10,50)},'historical_hit_differences':{str(k):0 for k in (1,5,10,50)},'category_metrics':{},'search_seconds':0.,'numpy_seconds':0.}
        for c in CATEGORIES:
            q,qids,_=dataset_query_vectors(label,c);index=INDEXES[label][c];frame=Q.set_index('query_id').loc[qids];lookup={v:i for i,v in enumerate(index.ids)};franks=[];nranks=[]
            batch=256 if label in HEADS else 32
            for start in range(0,len(q),batch):
                x=q[start:start+batch];rows=frame.iloc[start:start+batch];fs,fi,seconds=index.search_all(x);report['search_seconds']+=seconds
                ts=time.perf_counter();ns=x@index.vectors.T;report['numpy_seconds']+=time.perf_counter()-ts
                dense=np.empty_like(ns);np.put_along_axis(dense,fi,fs,axis=1);diff=np.abs(dense-ns);report['score_max_abs']=max(report['score_max_abs'],float(diff.max()))
                require(np.all(diff<=float32_dot_bound(x,index.vectors)),'FAISS score difference exceeds dimension-derived FP32 accumulation error bound')
                for i,row in enumerate(rows.itertuples()):
                    target=lookup[row.target_image_id];nr,no=rank_scores(ns[i],target);fr=int(np.flatnonzero(fi[i]==target)[0])+1;old=int(PRED[label].loc[row.Index]);franks.append(fr);nranks.append(nr)
                    item={'query_id':row.Index,'category':c,'faiss_rank':fr,'numpy_rank':nr,'historical_rank':old}
                    if fr!=nr:
                        others=np.flatnonzero((ns[i]>ns[i,target])!=(dense[i]>dense[i,target]));item.update(target_faiss_score=float(dense[i,target]),target_numpy_score=float(ns[i,target]),changed_candidates=[{'image_id':index.ids[j],'numpy_gap':float(ns[i,j]-ns[i,target]),'faiss_gap':float(dense[i,j]-dense[i,target]),'float64_gap':float(x[i].astype('float64')@(index.vectors[j]-index.vectors[target]).astype('float64'))} for j in others[:8]]);report['rank_differences'].append(item)
                    if not np.array_equal(fi[i,:10],no[:10]):report['top10_differences'].append({**item,'faiss_top10':[index.ids[j] for j in fi[i,:10]],'numpy_top10':[index.ids[j] for j in no[:10]],'boundary_gap_numpy':float(ns[i,no[9]]-ns[i,no[10]])})
                    if nr!=old:report['historical_rank_differences'].append(item)
                    for k in (1,5,10,50):report['metric_hit_differences'][str(k)]+=int((fr<=k)!=(nr<=k));report['historical_hit_differences'][str(k)]+=int((nr<=k)!=(old<=k))
            report['category_metrics'][c]={'faiss':{f'R@{k}':float((np.array(franks)<=k).mean()*100) for k in (1,5,10,50)},'numpy':{f'R@{k}':float((np.array(nranks)<=k).mean()*100) for k in (1,5,10,50)}};report['queries']+=len(q)
        report['reported_hits_equal']=not any(report['metric_hit_differences'].values());report['historical_reported_hits_equal']=not any(report['historical_hit_differences'].values());report['top10_identical']=not report['top10_differences'];report['all_ranks_identical']=not report['rank_differences']
        report['macro']={f'R@{k}':float(np.mean([v['faiss'][f'R@{k}'] for v in report['category_metrics'].values()])) for k in (1,5,10,50)};report['Mean']=(report['macro']['R@10']+report['macro']['R@50'])/2
        report['passed']=report['queries']==6016 and report['reported_hits_equal'] and report['historical_reported_hits_equal']
        DEMO_REPORT['full_faiss_validation'][label]=report;save_demo_reports()
        require(report['passed'],'FAISS/source Hit@K differs for '+label+'; full diagnostics saved; stop before accepting benchmark equivalence')
        print(label,'|6016 queries | Mean',round(report['Mean'],6),'| hits exact | rank drift',len(report['rank_differences']),'| top10 drift',len(report['top10_differences']))
    save_demo_reports();return DEMO_REPORT['full_faiss_validation']

def save_demo_reports():
    import importlib.metadata as md
    local=Path(CFG['local_cache']);local.mkdir(parents=True,exist_ok=True)
    CFG.setdefault('validation_run_id',time.strftime('demo_%Y%m%dT%H%M%S',time.gmtime())+'_'+uuid.uuid4().hex[:8])
    output=Path(CFG['index_root'])/'validation'/CFG['validation_run_id'];output.mkdir(parents=True,exist_ok=True)
    DEMO_REPORT['evidence_output']=str(output);HANDOVER['evidence_output']=str(output)
    DEMO_REPORT['runtime']['packages']={name:md.version(name) for name in ('torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow','faiss-cpu')}
    pip=Path('/content/taskb_demo_pip_check.txt');DEMO_REPORT['runtime']['pip_check']=pip.read_text() if pip.is_file() else 'not available'
    for name,value in [('test_report.json',DEMO_REPORT),('handover_manifest.json',compact_handover())]:
        data=canonical(value);(local/name).write_bytes(data)
        temporary=output/(name+'.'+uuid.uuid4().hex+'.partial');temporary.write_bytes(data);os.replace(temporary,output/name)


def compact_handover():
    value=copy.deepcopy(HANDOVER);sources=[]
    for row in value['sources']:
        binding=row['binding'];sources.append({'seed':row['seed'],'encoder':row['encoder'],'adapter':row['adapter'],'MR':row['MR'],'source_final_lock':binding['source_full_final_lock_sha256'],'MR_final_lock':binding['final_mr_lock_sha256'],'recipe':binding['recipe'],'training_data':{k:v for k,v in binding['data'].items() if k!='package'},'benchmark':row['benchmark'],'verification':'direct binary SHA and receipts; strict state loading seed42 only in this demo task'})
    value['sources']=sources
    for by_category in value['indexes'].values():
        for index in by_category.values():
            contract=index.pop('contract');index.update(dimension=contract['dimension'],dtype=contract['dtype'],count=contract['count'],mapping_sha256=contract['gallery_id_sha256'],vectors_sha256=contract['vectors_sha256'],metadata_path=str(Path(index['folder'])/'metadata.json'))
    return value


def infer_query(query_id,labels=None):
    require(query_id in set(Q.query_id),'Query ID not in official-val');row=Q.set_index('query_id').loc[query_id];result={}
    for label in (labels or [ADAPTER_LABEL]):
        require(label in MODELS,'Model unavailable: '+label);q,ids,encode_s=dataset_query_vectors(label,row.category);position=ids.index(query_id);index=INDEXES[label][row.category]
        batch=256 if label in HEADS else 32;start=position//batch*batch;sub=position-start
        scores,order,search_s=index.search_all(q[start:start+batch]);scores=scores[sub:sub+1];order=order[sub:sub+1];target=index.ids.index(row.target_image_id);rank=int(np.flatnonzero(order[0]==target)[0])+1
        result[label]={'rank':rank,'Hit@10':rank<=10,'Hit@50':rank<=50,'ids':[index.ids[j] for j in order[0,:10]],'search_seconds':search_s,'query_seconds':0.,'initial_category_encode_seconds':encode_s,'scores':scores[0,:10].tolist()}
    return result

_BASE_MODEL=None;_TOKENIZER=None;_PROCESSOR=None
@torch.inference_mode()
def native_pipeline():
    global _BASE_MODEL,_TOKENIZER,_PROCESSOR
    if _BASE_MODEL is None:
        from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor
        _BASE_MODEL=CLIPModel.from_pretrained(EXTRACTION_CONTRACT['encoder_id'],revision=EXTRACTION_CONTRACT['revision'],attn_implementation='eager',torch_dtype=torch.float32).eval().requires_grad_(False).to(DEVICE)
        require(tensor_hash(_BASE_MODEL)==EXTRACTION_CONTRACT['encoder_weight_sha256'],'Native Fashion-CLIP weights differ')
        _TOKENIZER=CLIPTokenizer.from_pretrained(EXTRACTION_CONTRACT['encoder_id'],revision=EXTRACTION_CONTRACT['revision'])
        _PROCESSOR=CLIPImageProcessor.from_pretrained(EXTRACTION_CONTRACT['encoder_id'],revision=EXTRACTION_CONTRACT['revision'])
        expected=EXTRACTION_CONTRACT['processor'];actual=_PROCESSOR.to_dict();require(all(actual.get(k)==v for k,v in expected.items() if k!='_processor_class'),'Native preprocessing differs')
    return _BASE_MODEL,_TOKENIZER,_PROCESSOR

@torch.inference_mode()
def custom_query(image,caption,category,labels=None):
    require(category in CATEGORIES,'Unknown category');require(isinstance(caption,str) and caption.strip('.?, ').strip(),'Caption rỗng/không có nội dung')
    require(isinstance(image,Image.Image),'Ảnh không đọc được');labels=labels or [ADAPTER_LABEL];require(all(x in MODELS for x in labels),'Model source unavailable')
    base,tokenizer,processor=native_pipeline();text=caption.strip('.?, ').capitalize() # one user-supplied modifier, no fabricated second caption
    tok=tokenizer([text],padding='max_length',max_length=77,truncation=False,return_tensors='pt');require(tok['input_ids'].shape==(1,77),'Caption quá77 token; không tự truncate')
    tok={k:v.to(DEVICE) for k,v in tok.items()};e=eos_positions(tok['input_ids'],tok['attention_mask'],tokenizer.eos_token_id);started=time.perf_counter()
    pixels=processor(images=image.convert('RGB'),return_tensors='pt').pixel_values.to(DEVICE,dtype=torch.float32);vision=base.vision_model(pixel_values=pixels);ref={'tokens':vision.last_hidden_state,'pooled':unit(base.visual_projection(vision.pooler_output))};results={}
    image_seconds=time.perf_counter()-started
    pretrained_h=None;trained=None;enhanced=None
    for label in labels:
        model_started=time.perf_counter()
        if label in HEADS:
            if pretrained_h is None:pretrained_h=base.text_model(**tok).last_hidden_state
            q=MODELS[label](pretrained_h,tok['attention_mask'],ref['tokens'])
        else:
            if trained is None:trained=TRAINED_ENCODER.encode_text(tok)
            h,m,e=trained
            if label==MR_LABEL:
                if enhanced is None:enhanced=frozen_adapter_text(MODELS[ADAPTER_LABEL],ref,h,m,e)
                reduced=reduce_frozen_text(enhanced,m,e,TRAINED_ENCODER.projection.weight);v=project_reference_tokens(ref['tokens'],base.vision_model.post_layernorm,base.visual_projection);q=MODELS[label](v,ref['pooled'],**reduced)
            else:q=MODELS[label].encode_query(ref,h,m,e)
        vectors=normalize_np(q.cpu().numpy());query_s=time.perf_counter()-model_started+(image_seconds if label==labels[0] else 0);scores,order,search_s=INDEXES[label][category].search_all(vectors);index=INDEXES[label][category]
        results[label]={'ids':[index.ids[j] for j in order[0,:10]],'scores':scores[0,:10].tolist(),'query_seconds':query_s,'search_seconds':search_s,'scope':'out-of-benchmark; no ground truth metric'}
    return text,results

def show_results(results,top_k,target=None):
    import matplotlib.pyplot as plt
    for label,r in results.items():
        if target is not None:print(label,'| target rank',r['rank'],'| Hit@10',r['Hit@10'],'| Hit@50',r['Hit@50'])
        else:print(label,'| Truy vấn ngoài benchmark, không có ground truth')
        started=time.perf_counter();fig,axes=plt.subplots(1,top_k,figsize=(2.4*top_k,3))
        for i,(ax,image_id) in enumerate(zip(axes,r['ids'][:top_k])):image_panel(ax,image_id,f'#{i+1}'+(' ★ TARGET' if image_id==target else ''))
        fig.suptitle(label);plt.tight_layout();plt.show();print('Seconds: query',round(r['query_seconds'],4),'search',round(r['search_seconds'],4),'display',round(time.perf_counter()-started,4))

def show_query(query_id,top_k=5,labels=None):
    require(top_k in (5,10),'Top-K must be5 or10');row=Q.set_index('query_id').loc[query_id];results=infer_query(query_id,labels)
    display(HTML('<b>'+html.escape(query_id)+'</b><br>Caption1: <code>'+html.escape(repr(row.caption_1))+'</code><br>Caption2: <code>'+html.escape(repr(row.caption_2))+'</code><br>Processed: <code>'+html.escape(row.modification_text)+'</code>'))
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(7,3));image_panel(axes[0],row.reference_image_id,'Reference');image_panel(axes[1],row.target_image_id,'Target annotation');plt.tight_layout();plt.show();show_results(results,top_k,row.target_image_id)

def decode_upload(content):
    require(isinstance(content,(bytes,bytearray,memoryview)) and len(content)>0,'Chưa upload ảnh')
    try:
        with Image.open(io.BytesIO(bytes(content))) as im:im.load();return im.convert('RGB')
    except Exception as e:raise RuntimeError('Ảnh upload không hợp lệ: '+type(e).__name__) from None

def launch_ui():
    import ipywidgets as w
    cat=w.Dropdown(options=CATEGORIES,description='Category');top=w.Dropdown(options=[5,10],description='Top-K');models=w.SelectMultiple(options=list(MODELS),value=(ADAPTER_LABEL,) if ADAPTER_LABEL in MODELS else (next(iter(MODELS)),),description='Models',rows=len(MODELS))
    qid=w.Text(description='Query ID');idx=w.BoundedIntText(value=0,min=0,max=2016,description='Index0+');group=w.Dropdown(options=['Tất cả','Adapter tốt hơn','Adapter kém hơn','Cả hai Top10','Cả hai ngoài Top50'],description='Nhóm');view=w.Button(description='Xem benchmark',button_style='primary');example=w.Button(description='Ví dụ seed cố định');upload=w.FileUpload(accept='image/*',multiple=False,description='Ảnh reference');caption=w.Textarea(description='Modifier');search=w.Button(description='Tìm ảnh mới',button_style='primary');out=w.Output()
    def changed(_):idx.max=len(eligible(cat.value,group.value))-1 if eligible(cat.value,group.value) else 0
    cat.observe(changed,names='value');group.observe(changed,names='value')
    def view_benchmark(_):
        with out:
            clear_output(wait=True)
            try:
                pool=eligible(cat.value,group.value);chosen=qid.value.strip() or pool[idx.value];require(chosen in pool,'Query ngoài category/nhóm');require(models.value,'Chọn ít nhất1 model');show_query(chosen,top.value,list(models.value))
            except Exception as e:print(type(e).__name__+': '+str(e))
    def examples(_):
        with out:clear_output(wait=True);print('Nhóm minh họa:',group.value,'|',sample_examples(group.value));print('Ví dụ chọn lọc không chứng minh cải thiện tổng thể.')
    def view_custom(_):
        with out:
            clear_output(wait=True)
            try:
                value=upload.value;item=next(iter(value.values())) if isinstance(value,dict) and value else value[0] if value else None;require(item is not None,'Upload1 ảnh reference');require(models.value,'Chọn model');image=decode_upload(item['content']);text,result=custom_query(image,caption.value,cat.value,list(models.value));display(image.resize((224,224)));print('Ngoài benchmark | modifier:',repr(text));show_results(result,top.value)
            except Exception as e:print(type(e).__name__+': '+str(e))
    view.on_click(view_benchmark);example.on_click(examples);search.on_click(view_custom)
    tabs=w.Tab(children=[w.VBox([w.HBox([qid,idx]),group,w.HBox([view,example])]),w.VBox([upload,caption,search])]);tabs.set_title(0,'Query benchmark');tabs.set_title(1,'Query tự nhập');display(w.VBox([w.HBox([cat,top]),models,tabs,out]))
    print('Hit theo annotation; ảnh khác target vẫn có thể phù hợp. Không so độ lớn cosine giữa model. Thiếu thumbnail không đổi gallery/rank.')


@torch.inference_mode()
def run_demo_smoke_tests():
    checks=[];before={label:tensor_hash(model) for label,model in MODELS.items()}
    for c in CATEGORIES:
        qid=Q[Q.category==c].iloc[0].query_id
        result=infer_query(qid,list(MODELS));row=Q.set_index('query_id').loc[qid]
        needed={row.reference_image_id,row.target_image_id}|{image for r in result.values() for image in r['ids']}
        for image_id in needed:
            paths=IMAGE_PATHS.get(image_id,set());require(len(paths)==1,'Missing/ambiguous test image:'+image_id)
            with Image.open(next(iter(paths))) as im:im.verify()
        checks.append({'category':c,'query_id':qid,'ranks':{label:r['rank'] for label,r in result.items()},'top_k':[5,10],'images_verified':len(needed)})
    # Exercise real image bytes + fresh caption with the same decoding function as FileUpload.
    first=Q.iloc[0];path=next(iter(IMAGE_PATHS[first.reference_image_id]));im=decode_upload(Path(path).read_bytes())
    text,results=custom_query(im,'make it blue with short sleeves',first.category,list(MODELS))
    require(all(len(r['ids'])==10 and 'rank' not in r and 'Hit@10' not in r for r in results.values()),'Custom query must not invent ground truth metrics')
    rejected=[]
    for caption in ('','   ','.?, '):
        try:custom_query(im,caption,'dress')
        except RuntimeError:rejected.append('empty_caption')
        else:raise RuntimeError('Empty caption was accepted')
    try:decode_upload(b'not an image')
    except RuntimeError:rejected.append('bad_image')
    else:raise RuntimeError('Corrupt upload was accepted')
    try:custom_query(im,'blue','dress',['nonexistent model'])
    except RuntimeError:rejected.append('missing_model')
    else:raise RuntimeError('Missing model was accepted')
    # Test mapping mismatch/corrupt index using temporary files; never touch actual indexes.
    import tempfile
    with tempfile.TemporaryDirectory() as folder:
        rng=np.random.default_rng(2026);v=normalize_np(rng.normal(size=(12,512)).astype('float32'));v[1]=v[0];ids=[str(i) for i in range(12)]
        fixture=ExactIPIndex('dress',ids,v,{'fixture':True},folder);_,order,_=fixture.search_all(v[:1]);require(order[0,:2].tolist()==[0,1],'FAISS tie fixture failed')
        reload=ExactIPIndex('dress',ids,v,{'fixture':True},folder);require(reload.key==fixture.key,'Index reload fingerprint differs')
        (fixture.folder/'mapping.json').write_text('["wrong"]')
        try:ExactIPIndex('dress',ids,v,{'fixture':True},folder)
        except RuntimeError:rejected.append('corrupt_mapping')
        else:raise RuntimeError('Corrupt index mapping accepted')
    require(before=={label:tensor_hash(model) for label,model in MODELS.items()},'Inference modified model weights')
    require(tensor_hash(TRAINED_ENCODER)==STORES['trained_text'].index['contract']['encoder_weights_sha256'] and all(not v.requires_grad for v in TRAINED_ENCODER.parameters()),'Native trained encoder changed/unfrozen')
    require(tensor_hash(_BASE_MODEL)==EXTRACTION_CONTRACT['encoder_weight_sha256'] and all(not v.requires_grad for v in _BASE_MODEL.parameters()),'Native pretrained vision/text encoder changed/unfrozen')
    DEMO_REPORT['smoke']={'passed':True,'queries':checks,'custom_query':{'real_reference_image_id':first.reference_image_id,'modifier':text,'models':list(results),'ground_truth_metrics':False,'decode_function_same_as_UI':True,'actual_browser_file_picker_tested':False},'rejected_cases':rejected,'model_weights_unchanged':True,'training_performed':False,'scope':'actual GPU inference, decoder and UI helper paths; browser controls separately checked'}
    save_demo_reports();print('Smoke passed:',len(checks),'categories | fresh image/caption',len(results),'models | rejected invalid inputs:',len(rejected));return DEMO_REPORT['smoke']


def register_query_index_bindings():
    # Query checkpoints do not produce gallery features. Register them separately,
    # preserving a shared, immutable gallery index when its exact contract matches.
    shared={}
    for label,by_category in INDEXES.items():
        for category,index in list(by_category.items()):
            if index.key in shared:
                other=shared[index.key];require(index.contract==other.contract and np.array_equal(index.vectors,other.vectors),'Cannot share mismatched gallery index');by_category[category]=other
            else:shared[index.key]=index
        if label in HEADS:
            pin=HEADS[label];binding={'model':label,'phase':'final','seed':42,'checkpoint':str(Path(pin['folder'])/'final_checkpoint.pt'),'checkpoint_sha256':pin['checkpoint_sha'],'config_sha256':pin['config_sha'],'text_source':'pretrained preparation cache'}
        else:
            binding={'model':label,'phase':'final','seed':42,'encoder':PAIR['encoder'],'text_index_sha256':PAIR['text_index_sha']}
            if label!= 'Conventional-only':binding['adapter']=PAIR['adapter']
            if label==MR_LABEL:binding['MR_checkpoint_sha256']=MR_RESULT['MR_final_sha256'];binding['MR_lock_sha256']=MR_RESULT['lock_sha256']
        binding.update(caption_policy=CAPTION_POLICY,reference_policy='retain in gallery (FashionIQ)',query_manifest_sha256=PREP_ANCHORS['official_val_queries.csv'])
        for category,index in by_category.items():
            value={**binding,'category':category,'gallery_index_fingerprint':index.key,'gallery_metadata':str(index.folder/'metadata.json')}
            path=index.folder/'query_bindings'/(fingerprint(value)+'.json');immutable_json(path,value);HANDOVER['indexes'][label][category]['query_binding']=str(path)
    HANDOVER['result_lock_verified_directly']=bool(DEMO_REPORT['locked_statistics'])


@torch.inference_mode()
def load_native_final_encoder():
    base,tokenizer,_=native_pipeline();asset=PAIR['encoder'];state=read_checkpoint(asset['path'],asset['sha256'])
    require(state['phase']=='final' and state['seed']==42 and state['kind']=='encoder' and state['finished'] and state['epoch']==3 and state['signature']==asset['signature'],'Native encoder provenance differs')
    model=ConventionalOnly(base.text_model,base.text_projection,tokenizer.eos_token_id)
    model.load_state_dict(state['model'],strict=True);model.freeze().to(DEVICE)
    require(tensor_hash(model)==STORES['trained_text'].index['contract']['encoder_weights_sha256'],'Native encoder weights differ from benchmark text cache')
    return model


## Demo
Chọn một hoặc nhiều mô hình. Index0-based trong category/nhóm; Query ID được ưu tiên. Nhóm ví dụ chỉ để minh họa, không chứng minh cải thiện tổng thể. Ảnh khác target vẫn có thể phù hợp về ngữ nghĩa. Ảnh hiển thị thiếu/hỏng được ghi ID/lỗi, không đổi gallery hoặc thứ hạng.

Query tự nhập chạy encoder/tokenizer/preprocessing thật; không dùng text cache benchmark cho caption mới. Setup có thể lâu do kiểm/copy cache Drive một lần và chuẩn bị ảnh nếu thiếu. Các lần query sau reuse model/index.


In [ ]:
setup_demo()
setup_mr_and_indexes()
if CFG['run_full_validation']: validate_faiss_all()
prepare_demo_images()
save_demo_reports()


In [ ]:
# Saved verification evidence belongs to this authorized inference run. No training.
RUN_UI_TESTS = False
register_query_index_bindings()
if RUN_UI_TESTS: run_demo_smoke_tests()
save_demo_reports()
validated=DEMO_REPORT.get('full_faiss_validation',{})
print('Final seed42 models:',len(MODELS),'| FAISS full-query checks:',{label:r['queries'] for label,r in validated.items()})
print('Native smoke:',DEMO_REPORT.get('smoke',{}).get('passed','not run'),'| Clean user-variable namespace:',DEMO_REPORT.get('clean_namespace',{}).get('passed','not run'))
launch_ui()


In [ ]:
# Optional: set True and run this cell to download the current manifest/test report.
EXPORT_REPORTS = False
if EXPORT_REPORTS:
    from google.colab import files
    save_demo_reports()
    package=Path(CFG['local_cache'])/'taskb_demo_handover.zip'
    with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ('handover_manifest.json','test_report.json'):
            archive.write(Path(CFG['local_cache'])/name,arcname=name)
    with zipfile.ZipFile(package) as archive:
        require(archive.testzip() is None,'Export archive integrity failed')
        for name in archive.namelist():json.loads(archive.read(name))
    files.download(str(package))
